# Experimento DFT × MobileNetV3 — versão corrigida

**O que mudou em relação à v1:**

1. `SpectralDepthwise` agora implementa o **teorema da convolução de verdade**:
   `y = IFFT(FFT(x) ⊙ K)` com kernel complexo aprendível, parte real da IFFT
   (não módulo).
2. **Inicialização do kernel a partir do depthwise pré-treinado** (flip espacial
   + roll para alinhar com a cross-correlation `same` do TF). No init, a camada
   é matematicamente equivalente ao depthwise original — testado abaixo.
3. **BatchNorm imediatamente após a camada espectral é descongelada**, para que
   a rede possa absorver pequenas variações nas estatísticas do feature map.
4. Variante de 3 blocos descartada — só comparamos `baseline` × `fourier_1block`.
5. Learning rate baixado para `1e-4` para não destruir a inicialização.

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, DepthwiseConv2D,
    BatchNormalization, Input,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score as sklearn_f1

print('TF version:', tf.__version__)

In [ ]:
# === Configuração ===
DISPOSITIVOS = ['dish washer', 'kettle', 'microwave', 'washing machine', 'fridge']
IMAGENS      = ['rp', 'gadf', 'gasf', 'mtf', 'tbg']
BATCH        = 32
FOLDER_I     = 'pickle_data'
N_RUNS       = 3
LR           = 1e-4   # menor que o padrão do Adam (1e-3); preserva o init espectral

# Variante de 3 blocos descartada (resultado anterior catastrófico).
VARIANTS = {
    'baseline':       0,
    'fourier_1block': 1,
}

## Camada `SpectralDepthwise` (corrigida)

**Conceito.** Pelo teorema da convolução, `(x * k)[n] = IFFT(FFT(x) · FFT(k))[n]`.
Em 2-D e por canal, fica:

$$y_c = \mathrm{IFFT}_2\bigl(\mathrm{FFT}_2(x_c) \odot K_c\bigr), \qquad
K_c \in \mathbb{C}^{H\times W}$$

`K` é **aprendível** (parte real e imaginária separadas como dois pesos `float32`).

**Equivalência com depthwise pré-treinado no init.** A convolução com FFT é
*circular*; a do TF (`DepthwiseConv2D` com `padding='same'`) é cross-correlation
linear. Para que `SpectralDepthwise(init=k_pretrained)` seja idêntica à
`DepthwiseConv2D(k_pretrained)` no início:

1. Espelhar o kernel espacialmente (`k[::-1, ::-1]`) — cross-corr ↔ conv.
2. Centrar via `np.roll(..., -(kh-1)//2, -(kw-1)//2)` — alinhar ao centro.
3. Zero-padar até `(H, W)` e tirar FFT 2-D.

Saídas no *interior* batem em ~1e-7 (precisão fp32). Erro de borda ~ tamanho do
kernel é desprezível para feature maps grandes.

In [ ]:
class SpectralDepthwise(tf.keras.layers.Layer):
    """
    Convolução depthwise via teorema da convolução:
        y_c = IFFT( FFT(x_c) ⊙ K_c )
    
    K é um kernel complexo aprendível (parte real e imaginária separadas)
    de shape (C, H, W). Pode ser inicializado a partir de um kernel
    DepthwiseConv2D pré-treinado, de modo que a camada começa
    funcionalmente equivalente ao conv original.
    
    Entrada: (B, H, W, C)
    Saída  : (B, H/sh, W/sw, C)   — stride por decimation no domínio espacial
    """
    def __init__(self, strides=(1, 1), pretrained_kernel=None, **kwargs):
        super().__init__(**kwargs)
        self.strides = (int(strides[0]), int(strides[1]))
        # numpy (kh, kw, C, 1) — usado só na build, depois descartado
        self._init_kernel = pretrained_kernel
    
    def build(self, input_shape):
        _, H, W, C = input_shape
        self.H, self.W, self.C = H, W, C
        
        if self._init_kernel is not None:
            kh, kw, c_in, dm = self._init_kernel.shape
            if c_in != C or dm != 1:
                raise ValueError(
                    f'Kernel shape {self._init_kernel.shape} incompatível '
                    f'com input C={C} (espera (kh, kw, {C}, 1))'
                )
            pad_h, pad_w = (kh - 1) // 2, (kw - 1) // 2
            
            # Cross-correlation 'same' do TF == circular conv com kernel
            # espacialmente espelhado e roll para o centro.
            k_flipped = self._init_kernel[::-1, ::-1, :, :]
            padded = np.zeros((C, H, W), dtype=np.float32)
            for c in range(C):
                padded[c, :kh, :kw] = k_flipped[:, :, c, 0]
            padded = np.roll(padded, shift=(-pad_h, -pad_w), axis=(-2, -1))
            
            K_complex = np.fft.fft2(padded, axes=(-2, -1))
            init_real = K_complex.real.astype(np.float32)
            init_imag = K_complex.imag.astype(np.float32)
        else:
            # Inicialização aleatória pequena (fallback / quando carrega de checkpoint)
            init_real = (np.random.randn(C, H, W) * 0.02).astype(np.float32)
            init_imag = (np.random.randn(C, H, W) * 0.02).astype(np.float32)
        
        self.kernel_real = self.add_weight(
            name='kernel_real', shape=(C, H, W),
            initializer=tf.keras.initializers.Constant(init_real),
            trainable=True,
        )
        self.kernel_imag = self.add_weight(
            name='kernel_imag', shape=(C, H, W),
            initializer=tf.keras.initializers.Constant(init_imag),
            trainable=True,
        )
        # Liberar memória do kernel de inicialização
        self._init_kernel = None
        super().build(input_shape)
    
    def call(self, x, training=None):
        # (B, H, W, C) -> (B, C, H, W) e cast para complex64
        x_t = tf.transpose(x, [0, 3, 1, 2])
        X = tf.signal.fft2d(tf.cast(x_t, tf.complex64))
        K_c = tf.complex(self.kernel_real, self.kernel_imag)   # (C, H, W)
        Y = X * K_c[None, ...]                                  # broadcast no batch
        y = tf.signal.ifft2d(Y)
        y = tf.cast(tf.math.real(y), tf.float32)                # parte real, NÃO módulo
        out = tf.transpose(y, [0, 2, 3, 1])                     # (B, H, W, C)
        sh, sw = self.strides
        if sh > 1 or sw > 1:
            out = out[:, ::sh, ::sw, :]
        return out
    
    def get_config(self):
        # pretrained_kernel não é serializado; pesos são salvos pelo Keras
        return {**super().get_config(), 'strides': self.strides}

### Sanity check: equivalência ao init

Compara `SpectralDepthwise` (inicializada com kernel `k`) ao `DepthwiseConv2D(k)`
sobre uma entrada aleatória. Espera-se erro relativo ~1e-7 no interior do mapa
(borda diverge por causa de circular vs linear convolution).

In [ ]:
# Sanity check: SpectralDepthwise(init=k) ≡ DepthwiseConv2D(k) no interior
np.random.seed(0)
H, W, C = 32, 32, 8
x_test = np.random.randn(2, H, W, C).astype(np.float32)

dw = DepthwiseConv2D(kernel_size=3, padding='same', use_bias=False, strides=1)
dw.build((None, H, W, C))
y_spatial = dw(x_test).numpy()

sdw = SpectralDepthwise(strides=(1, 1), pretrained_kernel=dw.weights[0].numpy())
y_spectral = sdw(x_test).numpy()

# Excluir 2 px de borda (efeito do circular vs zero-pad)
interior = (slice(None), slice(2, -2), slice(2, -2), slice(None))
abs_err = np.abs(y_spatial[interior] - y_spectral[interior]).max()
rel_err = abs_err / (np.abs(y_spatial[interior]).max() + 1e-9)

print(f'Erro absoluto máximo (interior): {abs_err:.2e}')
print(f'Erro relativo máximo (interior): {rel_err:.2e}')
assert rel_err < 1e-4, 'Init espectral diverge do depthwise — verificar implementação'
print('✓ SpectralDepthwise inicializa corretamente a partir do kernel depthwise')

## Construtores de modelo

`build_feature_extractor` substitui as primeiras `n_fourier` `DepthwiseConv2D`
por `SpectralDepthwise` (passando o kernel pré-treinado) e descongela a primeira
`BatchNormalization` que vem logo depois de cada camada espectral. O resto
permanece congelado em pesos do ImageNet.

In [ ]:
def build_feature_extractor(input_shape, n_fourier=0, name=None, verbose=False):
    """
    n_fourier=0 → MobileNetV3Large original, totalmente congelado
    n_fourier=N → primeiros N DepthwiseConv2D viram SpectralDepthwise
                  (inicializados do kernel pré-treinado).
                  BNs imediatamente seguintes ficam treináveis.
    """
    base = MobileNetV3Large(
        input_shape=input_shape, weights='imagenet', include_top=False,
    )
    
    if n_fourier == 0:
        for layer in base.layers:
            layer.trainable = False
        out = GlobalAveragePooling2D()(base.output)
        return Model(inputs=base.input, outputs=out,
                     name=name or 'FE_baseline')
    
    counter = [0]
    
    def clone_fn(layer):
        if isinstance(layer, DepthwiseConv2D):
            counter[0] += 1
            if counter[0] <= n_fourier:
                cfg = layer.get_config()
                strides = cfg.get('strides', (1, 1))
                weights = layer.get_weights()
                kernel = weights[0] if weights else None
                return SpectralDepthwise(
                    strides=strides,
                    pretrained_kernel=kernel,
                    name=f'sdw_{counter[0]}',
                )
        return layer
    
    cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)
    
    # Congelar tudo
    for layer in cloned.layers:
        layer.trainable = False
    
    # Descongelar SpectralDepthwise
    for layer in cloned.layers:
        if isinstance(layer, SpectralDepthwise):
            layer.trainable = True
            if verbose:
                print(f'  treinável: {layer.name} (espectral)')
    
    # Descongelar a primeira BN imediatamente após cada SpectralDepthwise
    layers_list = cloned.layers
    for i, layer in enumerate(layers_list):
        if isinstance(layer, SpectralDepthwise):
            for j in range(i + 1, min(i + 5, len(layers_list))):
                if isinstance(layers_list[j], BatchNormalization):
                    layers_list[j].trainable = True
                    if verbose:
                        print(f'  treinável: {layers_list[j].name} (BN após {layer.name})')
                    break
    
    out = GlobalAveragePooling2D()(cloned.output)
    return Model(inputs=cloned.input, outputs=out,
                 name=name or f'FE_fourier_{n_fourier}')


def build_full_model(input_shape, n_fourier=0, verbose=False):
    """Feature extractor + cabeça MLP idêntica ao notebook 1."""
    fe = build_feature_extractor(input_shape, n_fourier, verbose=verbose)
    x  = Dense(64, activation='relu')(fe.output)
    x  = Dropout(0.25)(x)
    x  = Dense(64, activation='relu')(x)
    x  = Dropout(0.25)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=fe.input, outputs=out,
                  name=f'model_fourier_{n_fourier}')
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=LR),
        metrics=['accuracy'],
    )
    return model

## Utilitários de benchmark

In [ ]:
def model_size_mb(model):
    n_params = sum(tf.size(w).numpy() for w in model.weights)
    return n_params * 4 / (1024 ** 2)


def count_params(model):
    trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    return trainable, non_trainable


def measure_latency_ms(model, x_sample, n_warmup=5, n_reps=20):
    for _ in range(n_warmup):
        _ = model(x_sample, training=False)
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = model(x_sample, training=False)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def load_data(disp, img, folder=FOLDER_I):
    """Splits 60/20/20 do notebook 1."""
    load = lambda fname: pickle.load(open(fname, 'rb'))
    X_tr = load(f'{folder}/X_{img}_train({disp}).pickle')
    y_tr = load(f'{folder}/y_train({disp}).pickle')
    X_va = load(f'{folder}/X_{img}_val({disp}).pickle')
    y_va = load(f'{folder}/y_val({disp}).pickle')
    X_te = load(f'{folder}/X_{img}_test({disp}).pickle')
    y_te = load(f'{folder}/y_test({disp}).pickle')
    return X_tr, y_tr, X_va, y_va, X_te, y_te

## Inspeção: blocos `DepthwiseConv2D` da MobileNetV3Large

Útil pra confirmar qual camada está sendo substituída e seu stride.

In [ ]:
_img0 = IMAGENS[0]
_dev0 = DISPOSITIVOS[0]
_X_sample = pickle.load(open(f'{FOLDER_I}/X_{_img0}_train({_dev0}).pickle', 'rb'))
_input_shape = _X_sample.shape[1:]
del _X_sample

_base = MobileNetV3Large(input_shape=_input_shape, weights='imagenet', include_top=False)
dw_info = [
    (i, l.name, l.get_config()['strides'], l.weights[0].shape)
    for i, l in enumerate(_base.layers)
    if isinstance(l, DepthwiseConv2D)
]
print(f'Input shape: {_input_shape}')
print(f'Total de blocos DepthwiseConv2D: {len(dw_info)}')
print(f"{'idx':>5}  {'nome':<48}  strides  kernel")
for idx, name, strides, kshape in dw_info[:5]:
    print(f'{idx:>5}  {name:<48}  {strides}    {tuple(kshape)}')
print('...')
del _base

## Smoke test do extrator com 1 bloco Fourier

Verifica que `build_feature_extractor(_, n_fourier=1, verbose=True)` retorna o
modelo esperado: 1 `SpectralDepthwise` + 1 BN treináveis, todo o resto congelado.

In [ ]:
print('--- Construindo extrator com 1 bloco Fourier (verbose) ---')
fe_test = build_feature_extractor(_input_shape, n_fourier=1, verbose=True)
n_train, n_frozen = count_params(fe_test)
print(f'\nParâmetros treináveis: {n_train:,}')
print(f'Parâmetros congelados: {n_frozen:,}')
print(f'Tamanho total       : {model_size_mb(fe_test):.2f} MB')
del fe_test
K.clear_session()

## Loop de experimentos

Para cada variante × dispositivo × tipo de imagem:
- Treina `N_RUNS` vezes
- Registra acurácia, F1-score, latência e tamanho

In [ ]:
os.makedirs('output', exist_ok=True)
results = []

for variant_name, n_fourier in VARIANTS.items():
    print(f"\n{'='*65}")
    print(f'VARIANTE: {variant_name}  ({n_fourier} blocos Fourier)')
    print(f"{'='*65}")
    
    for disp in DISPOSITIVOS:
        for img in IMAGENS:
            
            X_tr, y_tr, X_va, y_va, X_te, y_te = load_data(disp, img)
            input_shape = X_tr.shape[1:]
            
            run_accs, run_f1s, latencies = [], [], []
            size_mb, n_train, n_frozen = None, None, None
            
            for run in range(N_RUNS):
                ckpt = f'ckpt_dft_v2/{variant_name}/{disp}/{img}/run{run}/model.keras'
                os.makedirs(os.path.dirname(ckpt), exist_ok=True)
                
                model = build_full_model(input_shape, n_fourier)
                
                if run == 0:
                    size_mb  = model_size_mb(model)
                    n_train, n_frozen = count_params(model)
                
                model.fit(
                    X_tr, y_tr,
                    batch_size=BATCH,
                    epochs=100,
                    verbose=0,
                    validation_data=(X_va, y_va),
                    callbacks=[
                        EarlyStopping(
                            monitor='val_loss', patience=7,
                            restore_best_weights=False, verbose=0,
                        ),
                        ModelCheckpoint(
                            ckpt, monitor='val_accuracy',
                            save_best_only=True, verbose=0,
                        ),
                    ],
                )
                
                best = tf.keras.models.load_model(
                    ckpt,
                    custom_objects={'SpectralDepthwise': SpectralDepthwise},
                )
                
                preds = (best.predict(X_te, batch_size=BATCH, verbose=0) > 0.5).astype(int).flatten()
                run_accs.append(float(np.mean(preds == y_te)))
                run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))
                
                x_sample = X_te[:BATCH]
                latencies.append(measure_latency_ms(best, x_sample))
                
                del model, best
                K.clear_session()
            
            row = {
                'variant':          variant_name,
                'n_fourier':        n_fourier,
                'device':           disp,
                'image':            img,
                'acc_mean':         np.mean(run_accs),
                'acc_std':          np.std(run_accs),
                'f1_mean':          np.mean(run_f1s),
                'f1_std':           np.std(run_f1s),
                'latency_ms_mean':  np.mean(latencies),
                'latency_ms_std':   np.std(latencies),
                'model_size_mb':    size_mb,
                'trainable_params': n_train,
                'frozen_params':    n_frozen,
            }
            results.append(row)
            
            print(
                f'  {disp:<16} | {img:<5} | '
                f'acc={np.mean(run_accs):.3f}±{np.std(run_accs):.3f} | '
                f'f1={np.mean(run_f1s):.3f} | '
                f'lat={np.mean(latencies):.1f}ms'
            )

df_results = pd.DataFrame(results)
df_results.to_csv('output/dft_experiment_results_v2.csv', index=False)
print('\nResultados salvos → output/dft_experiment_results_v2.csv')
df_results.head()

## Resumo comparativo

In [ ]:
summary = (
    df_results
    .groupby('variant')[['acc_mean', 'f1_mean', 'latency_ms_mean',
                         'model_size_mb', 'trainable_params']]
    .mean()
    .round(4)
)
print(summary.to_string())

## Visualizações

### Heatmaps de acurácia (dispositivo × tipo de imagem)

In [ ]:
import matplotlib.pyplot as plt

n_variants = len(VARIANTS)
fig, axes = plt.subplots(1, n_variants, figsize=(6 * n_variants, 5))
if n_variants == 1:
    axes = [axes]

for ax, variant_name in zip(axes, VARIANTS):
    sub = df_results[df_results['variant'] == variant_name]
    pivot = sub.pivot(index='device', columns='image', values='acc_mean')
    im = ax.imshow(pivot.values, vmin=0.5, vmax=1.0, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'{variant_name}\n(acurácia média)')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if val < 0.8 else 'white')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('output/dft_v2_heatmaps_accuracy.png', dpi=150)
plt.show()

### Comparação de latência, acurácia e F1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

grouped = df_results.groupby('variant')

lat = grouped['latency_ms_mean'].mean()
axes[0].bar(lat.index, lat.values, color=['steelblue', 'darkorange'])
axes[0].set_title('Latência média (ms)')
axes[0].set_ylabel('ms / batch')
axes[0].tick_params(axis='x', rotation=15)

acc = grouped['acc_mean'].mean()
axes[1].bar(acc.index, acc.values, color=['steelblue', 'darkorange'])
axes[1].set_title('Acurácia média')
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=15)

f1 = grouped['f1_mean'].mean()
axes[2].bar(f1.index, f1.values, color=['steelblue', 'darkorange'])
axes[2].set_title('F1-score médio')
axes[2].set_ylim(0.5, 1.0)
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('output/dft_v2_benchmark_summary.png', dpi=150)
plt.show()

print('\nLatência por variante (ms):\n', lat.round(2).to_string())
print('\nAcurácia por variante:\n',     acc.round(4).to_string())
print('\nF1-score por variante:\n',     f1.round(4).to_string())

### Delta de acurácia: fourier_1block vs. baseline

In [ ]:
baseline_acc = df_results[df_results['variant'] == 'baseline'].set_index(['device', 'image'])['acc_mean']
fourier_acc  = df_results[df_results['variant'] == 'fourier_1block'].set_index(['device', 'image'])['acc_mean']

delta = (fourier_acc - baseline_acc).unstack('image')
print('--- Delta acc: fourier_1block − baseline ---')
print(delta.round(4).to_string())
print(f'\nMédia global: {delta.values.mean():+.4f}')
print(f'Casos onde fourier > baseline: {int((delta > 0).sum().sum())}/{int(delta.size)}')

## Próximos passos sugeridos

Se a versão de 1 bloco rodar OK, vale tentar (em ordem de prioridade):

1. **Aplicar a camada espectral em uma posição mais profunda** (ex: 5º ou 8º
   bloco depthwise) — o ganho conceitual de filtro global aprendível tende a
   ser maior em camadas com semântica mais abstrata.
2. **Variante GFNet** — kernel `(C, H, W)` aprendido do zero (sem init do
   pretrained), tipicamente substituindo *vários* blocos consecutivos. Exige
   descongelar mais da rede.
3. **Combinar com fine-tuning parcial** — descongelar não só a BN seguinte mas
   também o bloco IBR inteiro que contém a camada espectral.